# DCF Tear Sheet

**Use after source analytics:** Reporting notebooks render results produced by the pricing, analytics, statement, and portfolio notebooks; they do not replace those calculation workflows.

**Purpose:** Turn DCF model outputs into a compact valuation page with enterprise value, equity bridge, cash-flow forecast, and sensitivities.

**Prerequisites:** `04_statement_modeling/models/dcf_valuation.ipynb`.

**What you'll learn:**

- Provide DCF forecast and valuation inputs.
- Render `reporting.dcf_tearsheet` inline.
- Use WACC and terminal-growth sensitivities as reviewer context.

Project unlevered free cash flow, run a Gordon-growth DCF, and render a
`dcf_tearsheet` — the EV→equity bridge, the UFCF projection, a WACC /
terminal-growth sensitivity tornado, and a forecast summary.


In [1]:
import datetime as dt
import json

from finstack_quant import reporting
from finstack_quant.statements import Evaluator, ModelBuilder
from finstack_quant.statements_analytics import evaluate_dcf

qs = ["2025Q1", "2025Q2", "2025Q3", "2025Q4", "2026Q1", "2026Q2", "2026Q3", "2026Q4"]
b = ModelBuilder("acme-dcf")
b.periods("2025Q1..2026Q4", "2025Q2")
b.value("revenue", list(zip(qs, [100, 105, 110, 116, 122, 128, 134, 140])))
b.compute("ebitda", "revenue * 0.27")
b.compute("ufcf", "ebitda * 0.6")
b.compute("net_income", "ebitda * 0.5")
spec = b.build()
result = Evaluator().evaluate(spec)

# evaluate_dcf requires the model's currency in metadata.
raw = json.loads(spec.to_json())
raw.setdefault("meta", {})["currency"] = "USD"
model_json = json.dumps(raw)

TV = json.dumps({"type": "gordon_growth", "growth_rate": 0.025})
NET_DEBT = 51.0
WACC = 0.10

dcf = evaluate_dcf(model_json, WACC, TV, "ufcf", net_debt_override=NET_DEBT)
print("Enterprise value:", round(dcf["enterprise_value"], 1), " Equity value:", round(dcf["equity_value"], 1))


Enterprise value: 1117.1  Equity value: 1066.1


## WACC & terminal-growth sensitivity

Re-run the DCF at shifted WACC and terminal-growth assumptions and bridge the
change in equity value into a tornado (higher WACC ⇒ lower equity).

In [2]:
def equity_at(wacc, growth):
    tv = json.dumps({"type": "gordon_growth", "growth_rate": growth})
    return evaluate_dcf(model_json, wacc, tv, "ufcf", net_debt_override=NET_DEBT)["equity_value"]

base_eq = dcf["equity_value"]
sensitivity = [
    {
        "parameter_id": "WACC ±1%",
        "downside": equity_at(WACC + 0.01, 0.025) - base_eq,
        "upside": equity_at(WACC - 0.01, 0.025) - base_eq,
    },
    {
        "parameter_id": "Terminal growth ±0.5%",
        "downside": equity_at(WACC, 0.020) - base_eq,
        "upside": equity_at(WACC, 0.030) - base_eq,
    },
]
for s in sensitivity:
    print(f"{s['parameter_id']:26s} down={s['downside']:+.1f}  up={s['upside']:+.1f}")


WACC ±1%                   down=-131.2  up=+171.6
Terminal growth ±0.5%      down=-67.4  up=+77.1


## Valuation tear sheet

In [3]:
reporting.dcf_tearsheet(
    dcf,
    results=result,
    sensitivity=sensitivity,
    title="Acme Corp — DCF",
    generated=dt.date(2026, 6, 22),
)


,2025Q1,2025Q2,2025Q3,2025Q4,2026Q1,2026Q2,2026Q3,2026Q4
Revenue,100.00,105.00,110.00,116.00,122.00,128.00,134.00,140.00
EBITDA,27.00,28.35,29.70,31.32,32.94,34.56,36.18,37.80
Unlevered FCF,16.20,17.01,17.82,18.79,19.76,20.74,21.71,22.68
Net Income,13.50,14.18,14.85,15.66,16.47,17.28,18.09,18.90


## Saving a standalone HTML file

```python
ts = reporting.dcf_tearsheet(dcf, results=result, sensitivity=sensitivity, generated=dt.date(2026, 6, 22))
ts.save("dcf_tearsheet.html")
```


## Takeaways

- Reporting functions are presentation wrappers over analytics, valuation, statement, or portfolio results produced earlier in the curriculum.
- Keep the analytical source of truth in the typed objects or JSON specs, then render a tear sheet for review.
- Pass fixed `generated` dates in examples so notebook output remains reproducible.
